In [ ]:
from httpcore import stream
from prompt_toolkit.contrib.completers import system

from run_last_cell import add_assistant_message
from test_notebook import add_user_message, add_assistant_message
%pip install anthropic python-dotenv

In [ ]:
# Load env variables
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-opus-4-1"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text



In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [ ]:
print(final_answer)

In [ ]:
#New chat-bot
messages1 = []

while True:
    #Get user input
    user_input = input("> ")
    print(">",user_input)

    #Add user input to the list of messages
    add_user_message(messages1, user_input)
    #Call claude with the chat function
    response1 = chat(messages1)
    #Add generated text to the list of messages
    add_assistant_message(messages1, response1)
    #Print the generated text
    print(response1)

In [ ]:
messages2 = []
system="""
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step
"""
add_user_message(messages2, "How do i solve 5x+2=3 for x?")
answer = chat(messages2,system=system)
print(answer)

In [ ]:
messages3 = []

add_user_message(
    messages3,
    "Write a python function that checks a string for palindromes"
)

# Use a coding-focused system prompt for this task
coding_system = """
You are an expert Python programmer.
Write clean, efficient, and well-documented code.
Provide complete working solutions.
"""

answer = client.messages.create(
    model=model,
    max_tokens=1000,
    system=coding_system,
    messages=messages3
).content[0].text

print(answer)


In [ ]:
messages4 = []
add_user_message(messages4, "Generate a one sentence movie idea")
answer = chat(messages4, temperature=0.95)
answer

In [ ]:
messages5 = []
add_user_message(messages5, "Write a one sentence description of a fake database")
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages5,
    stream = True
)

for event in stream:
    print(event)

In [ ]:
messages5 = []
add_user_message(messages5, "Write a one sentence description of a fake database")
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages5,
) as stream:
    for text in stream.text_stream:
       # print(text, end="")
        pass

stream.get_final_message()

In [ ]:
messages6 = []

prompt = """Generate 3 different sample AWS cli commands and it should be very short"""

add_user_message(messages6, prompt)
add_assistant_message(messages6,"```bash")

text = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages6,
    stop_sequences=["```"]
).content[0].text

text.strip()